# 2. Exercise: Create your own multi-agent graph

Let's load the environment variables. They are automatically set up in the `settings`-instance imported below.
In the environment variables, we define the OpenAI API and Tavily API keys as well as the Langfuse credentials.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

# Construct src-path and append it to sys.path
src_path = Path.cwd().parent.parent
sys.path.append(str(src_path))

# Initialize settings-object that loads the OpenAI API and Tavily API keys + langfuse credentials to the environment
from core.settings import settings  # noqa: F401

## 2.a) Create your own agent

As the second exercise, we want to create our own agents specialized in creating Python charts (via `matplotlib`).
To build agents in Python, we will use the [`pydantic-ai`](https://pydantic.dev/docs/ai/overview/)-package, which supports Pydantic validation as a validation layer on top of the different agent SDKs such as the OpenAI SDK, the Google SDK, LangChain, LlamaIndex, AutoGPT, Transformers, CrewAI, Instructor and many more.

### Define prompts

First, let us define the system and user prompts of the agent.


<div class="alert alert-block alert-info">
<b>Note:</b> In the system prompt, we refer the `run_matplotlib` tool. It is a Python function provided by this repository that executes matplotlib code to create and save a figure.
</div>

In [ ]:
visualizer_system_prompt = """
You are the Chart Generator in a multi-Agent system.
- When the user asks for a chart, you MUST call the `run_matplotlib` tool.
- The `code` argument must be valid Python that uses the provided `plt` and `ax` to fully define the plot. Do NOT include imports or figure creation.
- The `filename` argument should be a short, snake_case description ending with '.png', e.g. 'revenue_per_quarter.png'.
- After the tool returns, briefly describe the chart and INCLUDE the file path in your answer.
- At the very end of your message, output EXACTLY two lines so the summarizer can find them:
    - FILE_PATH: <relative_path_to_chart_file>
    - CHART_NOTES: <one concise sentence summarizing the main insight in the chart>
- Do not include any other trailing text after these two lines.
"""

# TODO: Adjust the user query as you like, but it should indicate to create a chart
user_query = "Plot a straight line from 0 to 10."

### Initiate & Run your own agent

In the next three cells, we initiate, run and view the results of the visualization agent. Therefore, execute the following sub-tasks one after another:

1. Initiate and run the agent in its simplest configuration that plots a figure:
    - Add the function `run_matplotlib` as tool to the agent
    - Add the system prompt to the agent

2. Initiate and run the agent while specifying a specific output structure. Use the `Message` model from the module `models.state`. How does the output of the model change?

3. Create a custom model and use it as an output structure for the agent (instead of the `Message` model), e.g., extend the `Message` model by a `file_path` attribute. How does the output of the model change?

In [ ]:
from pydantic_ai import Agent

# from agents.visualizer.tools import run_matplotlib
# from models.state import Message


# TODO: initiate the agent. Minimalistic setup: add a tool to create a plot and add a system prompt
my_visualizer = Agent(
    model="openai:o3",
)

In [ ]:
# TODO: Run your agent with the user query
# response = ...
# print(response.output)

In [ ]:
from IPython.display import Image

# TODO: Adjust the file name if needed
display(Image("straight_line_0_to_10.png"))

## 2.b) Create your own graph

As our last exercise, we want to create a multi-agent graph architecture. This is done with [`pydantic-graph`](https://pydantic.dev/docs/ai/graph/graph/), which is an approach to model multi-agent workflows on top of `PydanticAI`. It is generally considered not as beginner-friendly as `PydanticAI`. In particular, `PydanticAI` itself provides several other approaches to model multi-agent workflows, besides using `pydantic-graph`. For further reading, check out, e.g., [Multi-Agent Patterns](https://pydantic.dev/docs/ai/guides/multi-agent-applications/).

**Graph:**
- It is a powerful abstraction to model, execute, control and visualize complex workflows.
- It is async with nodes and edges.

**GraphRunContext:**
- This is the context for the graph to run. It holds (a) the state of the graph and (b) dependencies and is passed to nodes when they're run.
- It is generic in the state of the graph it's used in `StateT`.

**Nodes:**
- Subclass of `BaseNode`, are dataclasses, and define the nodes for execution in the graph.
- They consist of:
    - fields containing any parameters required/optional when calling the node,
    - the business logic to execute the node, in the `run`-method,
    - return annotations of the `run`-method, which are read by `pydantic-graph` to determine the outgoing edges of the node.

### Create your own node that calls the visualizer

Now, we want to execute our first sub-task: Create a node that calls the `my_visualizer`-agent created in Exercise 2.a).
The following code block contains a suggested implementation of a `VisualizerNode`, that should be finalized by you.

In [ ]:
from pydantic import BaseModel
from pydantic_graph import BaseNode, GraphRunContext

from agents.executor import ExecutorNode
from agents.visualizer.prompts import visualizer_user_prompt
from core.logger import logger
from models.state import State


class MyVisualizerNode(BaseModel, BaseNode[State, None, None]):
    """VisualizerNode node built with pydantic-graph."""

    async def run(self, ctx: GraphRunContext[State]) -> ExecutorNode:
        """Run the visualizer node.

        Executes the visualization by creating `matplotlib` charts using the `run_matplotlib` tool.
        NOTE: This is the method containing the business logic. Here, the call to the `my_visualizer` agent and the update
        of the `State` instance.

        Args:
            ctx (GraphRunContext[State]): Gives the `State` instance as context to the node.

        Returns:
            End: Calls the `End` node after running the node.

        """
        logger.info("Invoke LLM in visualizer node to create a graph answering the user's query...")
        # TODO: 1) Initiate the user_prompt and 2) call the `my_visualizer` agent
        # NOTE: to refer to the `State` instance of the multi-agent graph, use the `ctx.state` attribute.
        # Here, this attribute refers to the `GraphRunContext` generic `StateT` that holds the state.
        # user_prompt =
        # response =
        # output = response.output

        logger.debug("Update multi-agent state...")
        # TODO: Append the multi-agent `state.messages` attribute with the output of the agent
        # NOTE: to update the `State` instance of the multi-agent graph, use the `ctx.state` attribute.

        # TODO: Tell the node the next node to go to. Here, we suggest going to the `ExecutorNode`.
        # return ...

### Initialize your own graph

Next, if we want to create a multi-agent graph with `pydantic-graph`, we need to have defined all required nodes, similarly to `MyVisualizerNode`.
This is done in this repository: `PlannerNode`, `ExecutorNode`, `WebResearchNode`, `ChartSummarizerNode` and `SynthesizerNode`.
Also, this repository supports a `VisualizerNode` - our suggested implementation of `MyVisualizerNode`.

In [ ]:
from pydantic_graph import Graph

# TODO: Import all required nodes
# For the `VisualizerNode`
#   - either import the `VisualizerNode` provided by this repository or
#   - use the `MyVisualizerNode` implemented by you above. Therefore, copy and overwrite the code in `agents.visualizer.visualizerNode` with your code. If doing so, you might restart this notebook
# from agents import ...

# TODO: initialize the graph object
# my_graph =

### Run your own graph

After initiating the graph object, as above, we need to define the `user_query` for the multi-agent, the list of `enabled_agents` and `my_state` - the initial state of the multi-agent.

In [ ]:
from models.agents import Agents
from models.state import Message, State

user_query = "Plot the current market capitalization of the top bank in Germany. State the day to which the final figure refers."
enabled_agents = [Agents.WEB_RESEARCHER, Agents.SYNTHESIZER, Agents.VISUALIZER, Agents.CHART_SUMMARIZER]

my_state = State(
    messages=[Message(content=user_query)],
    user_query=user_query,
    enabled_agents=enabled_agents,
)

Finally, we can run our graph...

In [ ]:
# run `my_graph` by awaiting its response

... and view its results.

In [ ]:
print(my_state.messages[-1].content)

In [ ]:
from IPython.display import Image

# TODO: Adjust the file name if needed
display(Image("germany_largest_bank_market_cap.png"))